# HDS-M03 Mini Project: Lung Segmentation from Chest X-rays

## Part 3: Further Processing and Assessment (steps 6–8)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# local helper module, all repeated code lives in lungseg.py
import lungseg

# create a folder for figures if it doesn't exist
os.makedirs('figures', exist_ok=True)

import pandas as pd
from scipy import ndimage as ndi
from scipy.stats import mannwhitneyu
from skimage import filters, morphology
from skimage.segmentation import clear_border

# create a folder for result tables if it doesn't exist
os.makedirs('results', exist_ok=True)

#### (6) Filtering and morphological operations

The Otsu mask from Part 2 still contains the air outside the body, holes and specks. Below, each
clean-up step is applied one at a time. This is exactly what `lungseg.segment_lungs` does.

In [ ]:
name      = lungseg.list_cases(label=0)[0]
image, gt = lungseg.load_case(name)

# store each intermediate result so it can be plotted afterwards
steps = {}

# blur away pixel noise
smooth = filters.gaussian(image, sigma=2)
steps['1. Gaussian smoothed'] = smooth

# Otsu threshold, dark = lung
binary = smooth < filters.threshold_otsu(smooth)
steps['2. Otsu threshold'] = binary

# opening cuts thin links and removes specks
binary = morphology.opening(binary, morphology.disk(5))
steps['3. Opening'] = binary

# remove the air outside the body, which touches the image border
binary = clear_border(binary)
steps['4. Clear border'] = binary

# keep the two largest regions (the lungs)
binary = lungseg.keep_largest_regions(binary, n=2)
steps['5. Two largest regions'] = binary

# fill holes inside the lungs
binary = ndi.binary_fill_holes(binary)
steps['6. Fill holes'] = binary

# closing smooths the outline
binary = morphology.closing(binary, morphology.disk(5))
steps['7. Closing (final)'] = binary

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, (title, im) in zip(axes.ravel(), steps.items()):

    ax.imshow(im, cmap='gray')

    # only the binary masks get a Dice score
    if im.dtype == bool:
        title += f'\nDice = {lungseg.dice(im, gt):.3f}'

    ax.set_title(title)

# last panel: final result (cyan) and ground truth (red) on the X-ray
axes[1, 3].imshow(image, cmap='gray')
axes[1, 3].contour(gt, levels=[0.5], colors='r')
axes[1, 3].contour(binary, levels=[0.5], colors='c')
axes[1, 3].set_title('red = ground truth, cyan = ours')

for ax in axes.ravel():
    ax.axis('off')

plt.tight_layout()
plt.savefig('figures/fig_3_pipeline_steps.pdf', bbox_inches='tight')
plt.show()

# sanity check that the package function gives the same result as the steps above
print(f'Same as lungseg.segment_lungs: {np.array_equal(binary, lungseg.segment_lungs(image))}')

A summary of the morphological operations (see the `biomedia_intro_04–07` practical notebooks):

- **Erosion** shrinks shapes, **dilation** grows them.
- **Opening** = erosion then dilation, which removes small objects.
- **Closing** = dilation then erosion, which fills small gaps.
- `disk(5)` is the structuring element, a circle of radius 5 pixels. Changing the radius changes how aggressive each step is.

#### (7) Using the ground truth for validation

The radiologist's masks allow each segmentation to be scored using two standard overlap measures
(`lungseg.dice` and `lungseg.jaccard`):

$$\text{Dice} = \frac{2|A\cap B|}{|A|+|B|}, \qquad \text{Jaccard} = \frac{|A\cap B|}{|A\cup B|}$$

Both are 1 for a perfect match and 0 for no overlap.

#### (8) Normal vs abnormal cases

I now run the pipeline on every normal and TB case. `N_PER_CLASS` allows a small subset to be used
while testing (`None` uses every image). Loading the full-size images is the slow part, so all 138
take a few minutes.

In [ ]:
# e.g. 10 for a quick test, None for all images
N_PER_CLASS = None

cases = lungseg.list_cases(0)[:N_PER_CLASS] + lungseg.list_cases(1)[:N_PER_CLASS]

# one dictionary per image, turned into a table at the end
rows = []

for name in cases:

    image, gt = lungseg.load_case(name)

    # Otsu only, and Otsu followed by the clean-up
    raw   = lungseg.segment_lungs(image, post_process=False)
    final = lungseg.segment_lungs(image)

    rows.append({
        'file'           : name,
        'class'          : 'TB' if lungseg.get_label(name) else 'normal',
        'dice_otsu_only' : lungseg.dice(raw, gt),
        'dice_final'     : lungseg.dice(final, gt),
        'jaccard_final'  : lungseg.jaccard(final, gt),
    })

results = pd.DataFrame(rows)
results.to_csv('results/segmentation_scores.csv', index=False)
results.head()

In [ ]:
# mean and standard deviation of each score, separately for normal and TB
score_cols = ['dice_otsu_only', 'dice_final', 'jaccard_final']
results.groupby('class')[score_cols].agg(['mean', 'std']).round(3)

In [ ]:
# distribution of Dice for each class, before and after the clean-up
data = [results.loc[results['class'] == c, col]
        for col in ['dice_otsu_only', 'dice_final'] for c in ['normal', 'TB']]

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(data)
ax.set_xticks([1, 2, 3, 4], ['Otsu only\nnormal', 'Otsu only\nTB', 'Final\nnormal', 'Final\nTB'])
ax.set_ylabel('Dice vs ground truth')
ax.set_title('Segmentation quality')
plt.savefig('figures/fig_3_dice_boxplot.pdf', bbox_inches='tight')
plt.show()

To check whether the difference between normal and TB cases is significant, I use a Mann-Whitney U
test, which compares two groups without assuming normally distributed scores.

In [ ]:
normal_dice = results.loc[results['class'] == 'normal', 'dice_final']
tb_dice     = results.loc[results['class'] == 'TB', 'dice_final']

stat, p = mannwhitneyu(normal_dice, tb_dice)

print(f'Mann-Whitney U = {stat:.1f}, p = {p:.3g}')

Finally, I look at the best and worst cases, since the numbers alone hide *why* the method fails.

In [ ]:
# 3 worst and 3 best cases by final Dice
ordered = results.sort_values('dice_final')
show    = list(ordered['file'].head(3)) + list(ordered['file'].tail(3))

fig, axes = plt.subplots(1, 6, figsize=(20, 4))

for ax, name in zip(axes, show):

    image, gt = lungseg.load_case(name)
    pred      = lungseg.segment_lungs(image)

    # red = radiologist, cyan = our method
    ax.imshow(image, cmap='gray')
    ax.contour(gt, levels=[0.5], colors='r')
    ax.contour(pred, levels=[0.5], colors='c')
    ax.set_title(f'{name}\nDice = {lungseg.dice(pred, gt):.2f}')
    ax.axis('off')

plt.suptitle('3 worst (left) and 3 best (right), red: ground truth, cyan: ours')
plt.savefig('figures/fig_3_best_worst.pdf', bbox_inches='tight')
plt.show()

Points for the write-up:

- How much did the clean-up improve Dice over Otsu alone?
- Is segmentation worse for TB cases, and why (bright lesions inside the lung, effusions, different lung shape)?
- Typical failure modes from the worst cases (lung joined to outside air, heart included, one lung lost).
- Limitations: one global threshold, fixed structuring-element sizes, no knowledge of lung shape.